In [ ]:
import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_v3.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)
ragas_df

In [5]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory(
    "gpt-4o-mini",
    provider="openai",
    client=async_client,
    max_tokens=4096,
    max_retries=6,
    timeout=120,  
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy,
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1516


statement: 蘇彥碩在使用Git時遇到的最大問題是，有人不會使用Git。
verdict: 1
reason: The context explicitly mentions that one of the major problems is that some people do not know how to use Git, which directly relates to 蘇彥碩's experience.


Evaluating:  10%|▉         | 19/200 [00:10<01:13,  2.45it/s]


statement: 在選擇SDGs主題時，需遵循以下原則。
verdict: 1
reason: The context outlines principles for selecting topics, including the SDGs, which supports this statement.

statement: 題目須排除過去三年的題目。
verdict: 1
reason: The context explicitly states that topics must exclude those from the past three years.

statement: 題目選擇應依據TronClass公告為準。
verdict: 1
reason: The context mentions that topic selection should be based on TronClass announcements.

statement: 應盡量選擇可以找到實際使用者進行訪談的題目。
verdict: 1
reason: The context advises to choose topics that allow for interviews with actual users.

statement: 根據SDGs，應挑選一個有興趣的SDG。
verdict: 1
reason: The context instructs to select an interesting SDG based on the SDGs.

statement: 從挑選的SDG中尋找主題。
verdict: 1
reason: The context indicates that one should find a topic from the selected SDG.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  14%|█▍        | 28/200 [00:14<00:57,  2.97it/s]

...


statement: 根據朱浩偉的經驗，朱浩偉在專題製作中建議要安排一個計畫。
verdict: 1
reason: 朱浩偉的經驗中提到要安排計畫以確保小組進度，因此這個陳述可以直接推斷。

statement: 安排計畫可以確保小組進度能夠跟著計劃進行。
verdict: 1
reason: 根據朱浩偉的經驗，安排計畫的確是為了確保小組進度，因此這個陳述是正確的。

statement: 朱浩偉建議把握每個可以寫小專案的機會。
verdict: 1
reason: 朱浩偉的經驗中提到要把握每一個可以寫小專案的機會，因此這個陳述是正確的。

statement: 釐清系統開發對使用者的價值會讓專題製作更順利。
verdict: 1
reason: 朱浩偉提到釐清系統開發對使用者的價值會使專題製作更順利，因此這個陳述是正確的。

statement: 至於時間投入方面，初期每週可能需要10-15小時。
verdict: 1
reason: 朱浩偉提到初期每週可能需要10-15小時的時間投入，因此這個陳述是正確的。

statement: 初期的時間主要用於研究和規劃。
verdict: 1
reason: 根據朱浩偉的經驗，初期的時間確實主要用於研究和規劃，因此這個陳述是正確的。

statement: 在進入開發和整合階段，接近死線或與產學合作方有審查時，每週投入的時間會增加到20-30小時甚至更多。
verdict: 1
reason: 朱浩偉提到在開發和整合階段，時間投入會增加到20-30小時甚至更多，因此這個陳述是正確的。


Evaluating: 100%|█████████▉| 199/200 [01:54<00:01,  1.16s/it]


statement: 林芯緹和蘇彥碩建議在共同開發時要事先溝通好程式的命名規則。
verdict: 1
reason: 根據林芯緹、蘇彥碩的經驗，確實提到要溝通好程式之命名規則，因此這個陳述可以直接推斷。

statement: 程式的命名規則最好以功能來命名。
verdict: 1
reason: 學姐建議以功能命名，這與陳述一致，因此可以直接推斷。

statement: 這樣可以讓所有人以及未來的自己都清楚程式碼的意義。
verdict: 1
reason: 文中提到可以將自己的邏輯想法寫在註解，讓大家和未來的自己都清楚程式碼的意義，因此這個陳述可以直接推斷。

statement: 另外，建議在開發過程中定期舉行「debug 大會」來追蹤組員的進度。
verdict: 1
reason: 文中提到偶爾開個debug 大會，這與陳述一致，因此可以直接推斷。

statement: 在「debug 大會」中，應該適時關心每位組員的狀況。
verdict: 1
reason: 文中提到並適時關心每一個組員進度，這與陳述一致，因此可以直接推斷。

statement: 這樣可以確保大家都在同一頁。
verdict: 1
reason: 雖然文中沒有明確提到這一點，但關心組員的狀況有助於確保大家在同一頁，因此可以推斷出這個陳述的合理性。

statement: 提高整體合作的效率是最終目標。
verdict: 1
reason: 文中提到要分工合作才能完成整個Project，這暗示了提高合作效率的重要性，因此可以推斷出這個陳述的合理性。


Evaluating: 100%|██████████| 200/200 [01:55<00:00,  1.73it/s]


statement: 根據蔡易辰的經驗，使用MySQL的優點是因為上學期（大二上）才剛學過。
verdict: 1
reason: 蔡易辰提到使用MySQL的原因是因為上學期剛學過，因此這個陳述可以直接推斷。

statement: 對於學生來說，使用MySQL比較熟悉。
verdict: 0
reason: 雖然蔡易辰提到使用MySQL的原因，但並未明確指出學生對MySQL的熟悉程度，因此無法直接推斷。

statement: 學生可以快速上手使用MySQL。
verdict: 0
reason: 蔡易辰提到上學期剛學過MySQL，但並未提到學生能否快速上手，因此無法直接推斷。

statement: 蔡易辰強烈建議不要只用MySQL這個資料庫。
verdict: 1
reason: 蔡易辰在上下文中明確提到不建議只用MySQL，因此這個陳述可以直接推斷。

statement: 在SA階段還有更多時間學習其他技術。
verdict: 1
reason: 蔡易辰提到在SA階段有更多時間學習其他程式語言，因此這個陳述可以直接推斷。

statement: 蔡易辰並未明確提到MySQL的缺點。
verdict: 1
reason: 蔡易辰在上下文中並未提到MySQL的缺點，因此這個陳述可以直接推斷。

statement: 如果專題要用另一個語言開發，可能會需要重新學習。
verdict: 1
reason: 蔡易辰提到如果專題要用另一個語言開發，會需要花時間從頭學習，因此這個陳述可以直接推斷。

statement: 這意味著若選擇MySQL而未掌握其他資料庫，轉換時可能會影響專題的進度。
verdict: 0
reason: 雖然蔡易辰提到需要學習其他技術，但並未明確提到選擇MySQL會影響專題進度，因此無法直接推斷。

statement: 目前沒有彭文暘的相關經驗資料。
verdict: 1
reason: 上下文中並未提到彭文暘的經驗，因此這個陳述可以直接推斷。

statement: 無法提供彭文暘的觀點。
verdict: 1
reason: 由於上下文中沒有提到彭文暘的觀點，因此這個陱述可以直接推斷。
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9

In [2]:
from pathlib import Path
import json
import ast
from datetime import datetime

import pandas as pd
import openai

from dotenv import load_dotenv

from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
from ragas.run_config import RunConfig

load_dotenv()


# =========================
# 1. 設定資料路徑
# =========================

TARGET_RUN_DIR = Path("eval_ragas_testset/20260604/v2")

INPUT_CSV = TARGET_RUN_DIR / "ragas_eval_with_response.csv"

# 輸出檔案
OUTPUT_SCORES_CSV = TARGET_RUN_DIR / "ragas_scores.csv"
OUTPUT_SUMMARY_MD = TARGET_RUN_DIR / "ragas_summary.md"
OUTPUT_SUMMARY_JSON = TARGET_RUN_DIR / "ragas_summary.json"


# =========================
# 2. 讀取 CSV
# =========================

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
# df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", nrows=1)

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# RAGAS 需要的欄位
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# =========================
# 3. 設定 LLM 與 Embedding
# =========================

async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory(
    "gpt-4o-mini",
    provider="openai",
    client=async_client,
    max_tokens=4096,
    max_retries=6,
    timeout=120,
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)
faithfulness = Faithfulness(llm=evaluator_llm)

faithfulness.statement_generator_prompt.instruction = """
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。

"""

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=ragas_embeddings,
)

answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
必須用繁體中文產生問題(除了專有名詞外)。"""


# =========================
# 4. 執行 RAGAS 評估
# =========================

scores = evaluate(
    dataset=rag_results,
    metrics=[
        faithfulness,
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy,
    ],
    run_config=RunConfig(
        max_workers=3,
        max_retries=6,
        timeout=120,
    ),
)

print(scores)


# =========================
# 5. 將結果轉成 DataFrame
# =========================

scores_df = scores.to_pandas()

# 加入原始問題與回答，方便之後檢查
result_df = pd.concat(
    [
        df.reset_index(drop=True),
        scores_df.reset_index(drop=True),
    ],
    axis=1,
)

# 輸出每題分數 CSV
result_df.to_csv(OUTPUT_SCORES_CSV, index=False, encoding="utf-8-sig")


# =========================
# 6. 計算平均分數
# =========================

metric_columns = [
    col
    for col in scores_df.columns
    if col not in ["user_input", "retrieved_contexts", "response", "reference"]
]

average_scores = scores_df[metric_columns].mean(numeric_only=True).to_dict()

question_count = len(df)

run_info = {
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target_run_dir": str(TARGET_RUN_DIR),
    "input_csv": str(INPUT_CSV),
    "question_count": question_count,
    "average_scores": average_scores,
    "output_scores_csv": str(OUTPUT_SCORES_CSV),
    "output_summary_md": str(OUTPUT_SUMMARY_MD),
}


# =========================
# 7. 輸出 JSON 摘要
# =========================

with open(OUTPUT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)


# =========================
# 8. 輸出好讀的 Markdown 摘要
# =========================

md_lines = []

md_lines.append("# RAGAS 評估結果摘要")
md_lines.append("")
md_lines.append(f"執行時間：`{run_info['run_time']}`")
md_lines.append("")
md_lines.append("## 資料來源")
md_lines.append("")
md_lines.append(f"- 評估資料夾：`{TARGET_RUN_DIR}`")
md_lines.append(f"- 輸入檔案：`{INPUT_CSV}`")
md_lines.append(f"- 題數：`{question_count}` 題")
md_lines.append("")
md_lines.append("## 平均分數")
md_lines.append("")

for metric_name, score_value in average_scores.items():
    md_lines.append(f"- `{metric_name}`：{score_value:.4f}")

md_lines.append("")
md_lines.append("## 輸出檔案")
md_lines.append("")
md_lines.append(f"- 每題詳細分數：`{OUTPUT_SCORES_CSV}`")
md_lines.append(f"- 摘要 JSON：`{OUTPUT_SUMMARY_JSON}`")
md_lines.append(f"- 摘要 Markdown：`{OUTPUT_SUMMARY_MD}`")
md_lines.append("")


with open(OUTPUT_SUMMARY_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))


print("評估完成！")
print(f"每題詳細分數已輸出：{OUTPUT_SCORES_CSV}")
print(f"摘要 Markdown 已輸出：{OUTPUT_SUMMARY_MD}")
print(f"摘要 JSON 已輸出：{OUTPUT_SUMMARY_JSON}")

C:\Users\User\AppData\Local\Temp\ipykernel_15200\2407803655.py:12: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15200\2407803655.py:12: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15200\2407803655.py:12: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15200\24078

instruction: 
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。



statement!!!: 林芯緹在使用Git時遇到的最大問題是有人不會用Git。
verdict: 1
reason: The context explicitly mentions that one of the biggest problems is that some people do not know how to use Git, which directly supports this statement.

statement!!!: 林芯緹建議能儘早學習使用Git。
verdict: 1
reason: The context suggests that it is advisable to learn Git as early as possible, which aligns with this statement.

statement!!!: 林芯緹建議在大家都還不熟悉的時候設定pull requests。
verdict: 1
reason: The context states that it is recommended to set up pull requests while everyone is still unfamiliar with Git, which directly supports this statement.

statement!!!: 設定pull requests可以讓大家都能確定程式碼無誤再進行最後的merge。
verdict: 1
reason: The context explains that setting up pull requests allows everyone to ensure the code is correct b

Evaluating:   4%|▍         | 8/200 [00:16<04:49,  1.51s/it]

instruction: 
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。



statement!!!: 在系統實作的 demo 階段，學生常遇到的問題包括時間不足。
verdict: 1
reason: 根據蔡易辰的經驗，提到時間不夠是實作過程中的一個問題，因此這個陳述可以直接推斷出來。

statement!!!: 學生常面臨某些功能開發時間過長的情況，導致無法如期完成整體系統的開發。
verdict: 1
reason: 文中提到某個功能可能做特別久，這表明學生在開發過程中可能會面臨時間不足的情況，因此這個陳述是可以推斷的。

statement!!!: 在實作過程中，學生可能會突然想要加入新功能，然而之前已經撰寫好的文件需要進行調整，這會增加額外的工作量。
verdict: 1
reason: 文中提到學生在實作過程中可能會想加某個功能，並且需要花額外的時間修改文件，因此這個陳述是正確的。

statement!!!: 由於整合及除錯的時間常常不足，學生在接近發表時仍會嘗試新增功能，這使得最後的準備變得非常匆忙。
verdict: 1
reason: 根據董書妤、周佳欣的經驗，提到整合時間匆忙和在快要發表時新增功能的情況，因此這個陳述是可以推斷的。

statement!!!: 系統運作的流程需要經過詳細討論，但當團隊成員意見不一致或討論不夠時，需要重新設計，浪費了寶貴的時間。
verdict: 1
reason: 文中提到系統運作的流程需要討論，且意見不一致時可能需要重新設計，因此這個陳述是正確的。

statement!!!: 這些問題都可能影響最終的 demo 成果。
verdict: 0
reason: 雖然文中提到的問題可能影響整體開發，但並未明確指出這些問題會影響最終的 demo 成果，因此這個陳述無法直接推斷。


Evaluating:   6%|▌         | 11/200 [00:23<05:22,  1.70s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

instruction: 
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。



statement!!!: 在第四章中，使用者回饋的蒐集方式包括測試後訪談、Google 問卷、和使用紀錄觀察等方式。
verdict: 1
reason: 根據上下文，使用者回饋的蒐集方式確實包括測試後訪談、Google 問卷和使用紀錄觀察，因此這個陳述可以直接推斷。

statement!!!: 回饋分析包含量化數據分析和質化回饋彙整。
verdict: 1
reason: 上下文中提到回饋分析應包含量化數據分析和質化回饋彙整，因此這個陳述是正確的。

statement!!!: 回饋分析具體為整理問卷分數、滿意度與任務完成率等數據。
verdict: 1
reason: 上下文中提到回饋分析應包含整理問卷分數、滿意度和任務完成率等數據，因此這個陳述可以直接推斷。

statement!!!: 回饋分析還包括使用者的意見、痛點、建議與實際感受。
verdict: 1
reason: 上下文中提到回饋分析應包含使用者的意見、痛點、建議與實際感受，因此這個陳述是正確的。

statement!!!: 團隊需要綜合這些回饋得出洞見。
verdict: 1
reason: 上下文中提到團隊應綜合回饋後，說明從中學到了什麼，因此這個陳述可以直接推斷。

statement!!!: 團隊針對回饋提出已做的修改或調整。
verdict: 1
reason: 上下文中提到團隊針對回饋說明已做了哪些修改或調整，因此這個陳述是正確的。

statement!!!: 這些回饋對於系統的未來展望影響深遠。
verdict: 1
reason: 上下文中提到未來展望是根據實作結果、測試結果與使用者回饋後，提出後續可能的開發方向，因此這個陳述可以直接推斷。

statement!!!: 這些回饋能幫助團隊了解使用者需求與痛點。
ver

Evaluating:  98%|█████████▊| 196/200 [06:26<00:09,  2.41s/it]

instruction: 
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。



statement!!!: 在需求分析中，透過利害關係人分析，可以確定系統的市場定位與利基。
verdict: 0
reason: 雖然利害關係人分析有助於理解需求，但文中並未明確指出其能直接確定市場定位與利基，因此無法直接推斷。

statement!!!: 首先需要針對與系統相關的所有利害關係人進行調查，這包括使用者、管理者、投資者等。
verdict: 0
reason: 文中提到需列出所有與系統相關的人，並分析不同利害關係人，但未具體提到調查的對象，因此無法直接推斷。

statement!!!: 對於不同的利害關係人，分析他們對系統的需求和期待，有助於理解不同群體的重要需求與痛點。
verdict: 1
reason: 文中提到需分析不同利害關係人對系統的期待差異，這與該陳述相符，因此可以直接推斷。

statement!!!: 使用者可能更注重功能的便利性。
verdict: 1
reason: 文中提到不同利害關係人對系統的期待差異，雖然未明確指出使用者的重點，但可以合理推斷使用者會關注功能的便利性，因此可以推斷。

statement!!!: 管理者則可能關注成本效益。
verdict: 1
reason: 文中提到不同利害關係人對系統的期待差異，雖然未明確指出管理者的重點，但可以合理推斷管理者會關注成本效益，因此可以推斷。

statement!!!: 將各利害關係人的需求進行整理，找出共通點與差異點，可以幫助定位市場並找到未被滿足的需求。
verdict: 1
reason: 文中提到整理利害關係人的需求有助於理解不同群體的重要需求與痛點，這與該陳述相符，因此可以直接推斷。

statement!!!: 結合利害關係人分析所得的資訊，界定系統的獨特價值主張。
verdict: 1
reaso

Evaluating: 100%|██████████| 200/200 [06:33<00:00,  1.97s/it]

instruction: 
給定一個問題與一個回答，請分析回答內容，並且只抽取可以根據 context 檢查的事實性、可驗證主張。
請忽略問候語、結語、禮貌用語、主觀鼓勵、建議語，以及像「希望以上內容對你有幫助」、「如果還有問題可以再詢問」、「你可以依照自己的情況調整」這類非事實性句子。
請將每個具有事實性的句子拆解成一個或多個可以完整理解的 statements。
每個 statement 不要使用代名詞，必須能獨立理解。
請用 JSON 格式輸出。



statement!!!: 朱浩偉在製作專題時會安排一個計畫以確保小組的進度能跟著計劃走。
verdict: 1
reason: 根據朱浩偉的經驗，他提到要盡量安排出一個計畫，確保小組進度能跟著計劃的進度走。

statement!!!: 朱浩偉強調釐清系統開發對使用者的價值。
verdict: 1
reason: 朱浩偉提到釐清系統開發對於使用者的價值會使製作專題更順利，因此可以推斷他強調這一點。

statement!!!: 朱浩偉會詢問並關心每個組員目前的進度以了解他們遇到的問題。
verdict: 1
reason: 朱浩偉提到會盡量詢問關心每個組員的進度，並了解他們遇到的問題，因此這一點可以直接推斷。

statement!!!: 朱浩偉會分享如果是他的話會如何撰寫或者進行開發。
verdict: 1
reason: 朱浩偉提到會告訴組員如果是自己的角度會怎麼寫，因此可以推斷他會分享這些內容。

statement!!!: 朱浩偉會運用網上資源進行查詢。
verdict: 1
reason: 朱浩偉提到會運用網上資源查詢，因此這一點可以直接推斷。

statement!!!: 朱浩偉會告訴大家如何總結並開始開發。
verdict: 1
reason: 朱浩偉提到會總結出如何開發後再告訴大家，因此這一點可以直接推斷。

statement!!!: 朱浩偉強調專題的重要性以保持大家的動力和效率。
verdict: 1
reason: 朱浩偉提到要讓每個人認知到專題的重要性以保持動力，因此可以推斷他強調這一點。

statement!!!: 朱浩偉促進共識的討論。
verdict: 1
reason: 朱浩偉提到要有效率及共識的討論，因此可以推斷他促進這樣的討論。
{'faithfulness': 0.85

In [30]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory("gpt-4o-mini", provider="openai", client=async_client,max_tokens=4096)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1195

{'faithfulness': 0.7917, 'context_precision': 0.9615, 'context_recall': 0.9038, 'answer_relevancy': 0.9091}


ragas_eval_with_response_v3
ragas_testset_v2
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

3.1
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

"ragas_eval_with_response_2026-05-22.csv"

{
    "faithfulness": 0.7917,
    "context_precision": 0.9615,
    "context_recall": 0.9038,
    "answer_relevancy": 0.9091,
}

ragas_eval_with_response_v3
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9333, 'answer_relevancy': 0.7727}